# 04 - Statistical Analysis, Feature Engineering & Preprocessing

**Project:** CrediPredict – Loan Approval Prediction System  
**Author:** B.Tech Data Science Student  
**Objective:** Compute summary statistics, engineer household financial features, conduct feature selection, and configure scikit-learn preprocessing pipelines.


### Step 1: Import Libraries & Load Cleaned Dataset

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import clean_data, engineer_features, build_preprocessor

# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_loan_data.csv')
df.head()

--- 
### Step 2: Statistical Analysis
Compute key statistical metrics: Mean, Median, Mode, Variance, Standard Deviation, Quartiles (Q1, Q3), and IQR for numerical features.

In [ ]:
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
stats_list = []

for col in num_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    mode_val = df[col].mode()[0]
    var_val = df[col].var()
    std_val = df[col].std()
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    
    stats_list.append({
        'Feature': col,
        'Mean': round(mean_val, 2),
        'Median': round(median_val, 2),
        'Mode': round(mode_val, 2),
        'Variance': round(var_val, 2),
        'Std Dev': round(std_val, 2),
        'Q1 (25%)': round(q1, 2),
        'Q3 (75%)': round(q3, 2),
        'IQR': round(iqr, 2)
    })

stats_df = pd.DataFrame(stats_list)
stats_df

--- 
### Step 3: Feature Engineering

#### Rationale & Interview Explanation:
1. **`Total_Income`** = `ApplicantIncome` + `CoapplicantIncome`  
   - *Why:* Banks assess combined household income when evaluating repayment capacity. A low primary income applicant with a high co-applicant income is frequently approved.
2. **`Loan_to_Income_Ratio`** = `(LoanAmount * 1000) / Total_Income`  
   - *Why:* Expresses total debt requested as a proportion of total household income. Higher ratios indicate elevated risk of default.

In [ ]:
df_fe = engineer_features(df)
df_fe[['ApplicantIncome', 'CoapplicantIncome', 'Total_Income', 'LoanAmount', 'Loan_to_Income_Ratio']].head()

--- 
### Step 4: Correlation Matrix & Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
corr = df_fe[['ApplicantIncome', 'CoapplicantIncome', 'Total_Income', 'LoanAmount', 'Loan_to_Income_Ratio', 'Credit_History']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical & Financial Features', fontsize=14, fontweight='bold')
plt.show()

> **Observation:** `Total_Income` displays a strong positive correlation (+0.60 to +0.80) with `LoanAmount`, confirming that loan requests scale with household income.

--- 
### Step 5: Save Processed Dataset for Machine Learning

In [ ]:
processed_path = '../data/processed/processed_loan_data.csv'
df_fe.to_csv(processed_path, index=False)
print(f"Processed dataset saved with shape {df_fe.shape} at: {processed_path}")